In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from torch.utils.data import DataLoader
from functools import partial

In [ ]:
cache_path = "./weights/huggingface"
model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
processor = AutoProcessor.from_pretrained(model_path, cache_dir=cache_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    _attn_implementation="flash_attention_2",
    cache_dir=cache_path
).to("cuda")

In [ ]:
def find_assistant_content_sublist_indexes(l):
    # (Pdb++) processor.tokenizer.encode("<|im_start|>assistant\n")
    # [151644, 77091, 198]
    # (Pdb++) processor.tokenizer.encode("<|im_end|>\n")
    # [151645, 198]

    start_indexes = []
    end_indexes = []

    # Iterate through the list to find starting points
    for i in range(len(l) - 1):
        # Check if the current and next elements form the start sequence
        if l[i] == 151644 and l[i+1] == 77091 and l[i+2] == 198:
            start_indexes.append(i+3)
            # Now look for the first 151645 and 198 after the start
            for j in range(i+3, len(l)-1):
                if l[j] == 151645 and l[j+1] == 198:
                    end_indexes.append(j+2)
                    break  # Move to the next start after finding the end

    return list(zip(start_indexes, end_indexes))

In [ ]:
from dataset_smolvlm.airsim_dataset import AirSimDataset, collate_fn
from dataset_smolvlm.vision_process import process_vision_info
train_dataset = AirSimDataset(["../data/train_data_sample.json"], 
                                  video_frame_num = 5,
                                  target_interval = 30)

In [ ]:
batch = [train_dataset.get_batch(7)]
masks_list = []
messages_list = []
traj_folders = []
target_times = []

for data in batch:
    masks_list.append(data["prob_map"])
    message = data["prob_message"]
    messages_list.append(message)
    traj_folders.append(data["traj_dir"])
    target_times.append(data["target_time"])


In [ ]:
texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in messages_list]

In [ ]:
image_inputs, video_inputs = process_vision_info(messages_list)

In [ ]:
inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

In [ ]:
input_ids_lists = inputs['input_ids'].tolist()


In [ ]:
labels = []
for ids_list in input_ids_lists:
    label_ids = [-100] * len(ids_list) # -100 is the ignore index in loss function
    for begin_end_indexs in find_assistant_content_sublist_indexes(ids_list):
        label_ids[begin_end_indexs[0]:begin_end_indexs[1]] = ids_list[begin_end_indexs[0]:begin_end_indexs[1]]
    labels.append(label_ids)
labels = torch.tensor(labels, dtype=torch.int64)

In [ ]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=2,
    collate_fn=partial(collate_fn, processor=processor, mission="prob"),
)

In [ ]:
for step, batch in enumerate(train_dataloader):
    print(batch)
    if step == 0:
        break

In [ ]:
batch[1]['masks_list'][0].shape

In [ ]:
batch[1].keys()

In [ ]:
import matplotlib.pyplot as plt

mask = batch[1]["masks_list"][0]          # torch.Size([1, 784, 784])
mask_2d = mask.squeeze(0).detach().cpu().numpy()  # (784, 784)

plt.figure(figsize=(6, 6))
plt.imshow(mask_2d, cmap="hot", vmin=0, vmax=1)
plt.colorbar()
plt.axis("off")
plt.show()


In [ ]:
type(batch[0])

In [ ]:
list(batch[0].keys())

In [ ]:
batch[0]['pixel_attention_mask'].shape

In [ ]:
batch[0]['input_ids'].shape

In [ ]:
batch[0]['attention_mask'].shape

In [ ]:
batch[0]['pixel_values'].shape

In [ ]:
import matplotlib.pyplot as plt

inputs, meta = next(iter(train_dataloader))
pv = inputs["pixel_values"]   # not batch[0], use inputs directly
print("pixel_values shape:", tuple(pv.shape))

img = pv
while img.dim() > 3:   # peel batch/image dims until CHW
    img = img[0]

# img is now (C, H, W)
img = img.detach().cpu().permute(1, 2, 0)
img = (img - img.min()) / (img.max() - img.min() + 1e-8)

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
plt.show()


In [ ]:
list(batch[0].keys()), list(batch[1].keys())


In [1]:
from dataset.airsim_dataset import AirSimDataset, collate_fn
from dataset.vision_process import process_vision_info
train_dataset = AirSimDataset(["../data/train_data_sample.json"], 
                                  video_frame_num = 5,
                                  target_interval = 30)

dataset has 10588 samples load from file ['../data/train_data_sample.json']


In [2]:
from models.processing_qwen2_vl import Qwen2VLProcessor
from models.image_processing_qwen2_vl import Qwen2VLImageProcessor
from models.processing_qwen2_vl import Qwen2VLProcessor
from models.image_processing_qwen2_vl import Qwen2VLImageProcessor
from models.pilot_llm import PilotLLM

from PIL import Image
import requests
import torch
from torchvision import io
from typing import Dict
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from torch.utils.data import DataLoader
from functools import partial

# Load the model in half-precision on the available device(s)
# modelq = Qwen2VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen2-VL-2B-Instruct", torch_dtype="auto", device_map="auto",_attn_implementation="flash_attention_2", cache_dir=cache_path
# )
# processorq = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct", cache_dir=cache_path)

processorf = Qwen2VLProcessor.from_pretrained("/storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/pilot_llm/weights/huggingface/models--Qwen--Qwen2-VL-2B-Instruct/snapshots/895c3a49bc3fa70a340399125c650a463535e71c", padding_side="right")
modelf = PilotLLM.from_pretrained("/storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/pilot_llm/weights/huggingface/models--Qwen--Qwen2-VL-2B-Instruct/snapshots/895c3a49bc3fa70a340399125c650a463535e71c").to('cuda')

/storage/project/r-cj124-0/sibidapo3/anxcnda/aeroduo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.93it/s]
Some weights of PilotLLM were not initialized from the model checkpoint at /storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/pilot_llm/weights/huggingface/models--Qwen--Qwen2-VL-2B-Instruct/snapshots/895c3a49bc3fa70a340399125c650a463535e71c and are newly initialized: ['depth_head.mlp.0.bias', 'depth_head.mlp.0.weight', 'depth_head.mlp.2.bias', 'depth_head.mlp.2.weight', 'mask_head.mlp.0.bias', 'mask_head.mlp.0.weight', 'mask_head.mlp.2.bias', 'mask_head.mlp.2.weigh

In [3]:
train_dataloaderf = DataLoader(
    train_dataset,
    batch_size=2,
    collate_fn=partial(collate_fn, processor=processorf, mission="prob"),
)

In [4]:
for step, batchf in enumerate(train_dataloaderf):
    print(batchf)
    if step == 0:
        break

({'input_ids': tensor([[151644,   8948,    198,  ..., 151653, 151645,    198],
        [151644,   8948,    198,  ..., 151653, 151645,    198]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]]), 'pixel_values': tensor([[-1.7923, -1.7923, -1.7923,  ..., -1.4802, -1.4802, -1.4802],
        [-1.7923, -1.7923, -1.7923,  ..., -1.4802, -1.4802, -1.4802],
        [-1.7923, -1.7923, -1.7923,  ...,  1.5487, -1.4802, -1.4802],
        ...,
        [-1.7923, -1.7923, -1.7923,  ..., -1.4802, -1.4802, -1.4802],
        [-1.7923, -1.7923, -1.7923,  ..., -1.4802, -1.4802, -1.4802],
        [-1.7923, -1.7923, -1.7923,  ..., -1.4802, -1.4802, -1.4802]]), 'image_grid_thw': tensor([[ 1, 56, 56],
        [ 1, 56, 56],
        [ 1, 56, 56],
        [ 1, 56, 56]])}, {'masks_list': [tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0.,

In [5]:
list(batchf[0].keys()), list(batchf[1].keys())


(['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw'],
 ['masks_list',
  'labels',
  'mission',
  'image_inputs',
  'traj_folders',
  'pred_time'])

In [6]:
batchf[0]['input_ids'].shape

torch.Size([2, 1788])

In [7]:
batchf[0]['attention_mask']

tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])

In [8]:
batchf[1]['labels']

tensor([[  -100,   -100,   -100,  ..., 151653, 151645,    198],
        [  -100,   -100,   -100,  ..., 151653, 151645,    198]])

In [9]:
modelf.config.output_hidden_states

False

In [10]:
batchf[1]['masks_list'][0].shape

torch.Size([1, 784, 784])

In [11]:
processorf

Qwen2VLProcessor:
- image_processor: Qwen2VLImageProcessor {
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_processor_type": "Qwen2VLImageProcessor",
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "max_pixels": 12845056,
  "merge_size": 2,
  "min_pixels": 3136,
  "patch_size": 14,
  "processor_class": "Qwen2VLProcessor",
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "longest_edge": 12845056,
    "shortest_edge": 3136
  },
  "temporal_patch_size": 2
}

- tokenizer: Qwen2TokenizerFast(name_or_path='/storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/pilot_llm/weights/huggingface/models--Qwen--Qwen2-VL-2B-Instruct/snapshots/895c3a49bc3fa70a340399125c650a463535e71c', vocab_size=151643, model_max_length=32768, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|

In [12]:
modelf.model.config

Qwen2VLConfig {
  "architectures": [
    "Qwen2VLForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "image_token_id": 151655,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "max_position_embeddings": 32768,
  "max_window_layers": 28,
  "model_type": "qwen2_vl",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_scaling": {
    "mrope_section": [
      16,
      24,
      24
    ],
    "rope_type": "default",
    "type": "default"
  },
  "rope_theta": 1000000.0,
  "sliding_window": 32768,
  "text_config": {
    "architectures": [
      "Qwen2VLForConditionalGeneration"
    ],
    "attention_dropout": 0.0,
    "bos_token_id": 151643,
    "eos_token_id": 151645,
    "hidden_act": "silu",
    "hidden_size": 1536,
    "image_token_id": null,
    "initializer_range": 0.02,
    "intermediate_size": 8960,
   

In [13]:

# # load pretrain model（seg and depth pretraining
# if args.reload_model_path is not None:
#     model = PeftModel.from_pretrained(model, args.reload_model_path)
#     weights = torch.load(os.path.join(args.reload_model_path, "pytorch_model/mp_rank_00_model_states.pt"))
#     model.load_state_dict(weights['module'], strict=False)

#     # If loading a trained main model, do not rewrite a set of lora
#     model = model.merge_and_unload()
#     if accelerator.is_main_process:
#         del weights

from peft import get_peft_model, LoraConfig, PeftModel
lora_config = LoraConfig(
    r=8,
    target_modules=['down_proj','o_proj','k_proj','q_proj','gate_proj','up_proj','v_proj'],
    task_type="CAUSAL_LM",
)
modelf = get_peft_model(modelf, lora_config)

In [14]:
modelf.print_trainable_parameters()

trainable params: 9,232,384 || all params: 2,221,784,578 || trainable%: 0.4155391162319968


In [15]:
for n, p in modelf.named_parameters():
    if any(
        [
            x in n
            for x in ["pooling_head", "depth_head", 
                        "seg_head", "seg_query", 
                        "coord_head", "coord_query"]
        ]
    ):
        print("n: ", n, "p.shape: ", p.shape)
        p.requires_grad = True

n:  base_model.model.depth_head.mlp.0.weight p.shape:  torch.Size([768, 1536])
n:  base_model.model.depth_head.mlp.0.bias p.shape:  torch.Size([768])
n:  base_model.model.depth_head.mlp.2.weight p.shape:  torch.Size([1, 768])
n:  base_model.model.depth_head.mlp.2.bias p.shape:  torch.Size([1])


In [16]:
batchf[0]['image_grid_thw'].shape

torch.Size([4, 3])

In [17]:
import torch.nn as nn
num_image_token = 784
mask_query = nn.Parameter(
    torch.zeros([num_image_token, modelf.config.hidden_size])
).to('cuda')

In [18]:
mask_query.shape

torch.Size([784, 1536])

In [19]:
import torch.nn as nn
import torch.nn.functional as F
class MaskHead(nn.Module):
    def __init__(self, hidden_size, upsample_mode='bilinear'):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1),
        )
        self.hidden_size = hidden_size
        self.upsample_mode = upsample_mode

    def forward(self, mask_hidden_states, thw):
        bs, image_token_num, hidden_size = mask_hidden_states.shape
        t, patch_h, patch_w = thw
        h = w = image_token_num
        mask_hidden_states = self.mlp(mask_hidden_states)
        mask_hidden_states = mask_hidden_states.view(bs, 1, patch_h//2, patch_w//2)
        mask_hidden_states = F.interpolate(mask_hidden_states, size=(patch_h, patch_w), 
                                           mode=self.upsample_mode, align_corners=False)
        pred_masks = F.interpolate(mask_hidden_states, size=(h, w), 
                                   mode=self.upsample_mode, align_corners=False)
        return pred_masks

In [20]:
input_ids = batchf[0]['input_ids'].to('cuda')
attention_mask = batchf[0]['attention_mask'].to('cuda')
position_ids = None
past_key_values = None
inputs_embeds = None
labels = batchf[1]['labels'].to('cuda')
masks_list = batchf[1]['masks_list']
use_cache = None
output_attentions =  None
output_hidden_states =  None,
return_dict = None
pixel_values = batchf[0]['pixel_values'].to('cuda')
pixel_values_videos = None,
image_grid_thw = batchf[0]['image_grid_thw'].to('cuda')
video_grid_thw = None,
rope_deltas = None,
cache_position = None,
is_inference = False,

In [21]:
output_attentions = output_attentions if output_attentions is not None else modelf.model.config.output_attentions
output_hidden_states = (
    output_hidden_states if output_hidden_states is not None else modelf.model.config.output_hidden_states
)
return_dict = return_dict if return_dict is not None else modelf.model.config.use_return_dict

bs = input_ids.shape[0]
labels = labels.to(input_ids.device)

In [22]:
if inputs_embeds is None:
    inputs_embeds = modelf.model.model.embed_tokens(input_ids)
    if pixel_values is not None:
        pixel_values = pixel_values.type(modelf.visual.get_dtype())
        with torch.no_grad():
            image_embeds = modelf.visual(pixel_values, grid_thw=image_grid_thw)
        n_image_tokens = (input_ids == modelf.model.config.image_token_id).sum().item()
        n_image_features = image_embeds.shape[0]
        if n_image_tokens != n_image_features:
            raise ValueError(
                f"Image features and image tokens do not match: tokens: {n_image_tokens}, features {n_image_features}"
            )
        image_mask = (
            (input_ids == modelf.model.config.image_token_id)
            .unsqueeze(-1)
            .expand_as(inputs_embeds)
            .to(inputs_embeds.device)
        )
        image_embeds = image_embeds.to(inputs_embeds.device, inputs_embeds.dtype)
        inputs_embeds = inputs_embeds.masked_scatter(image_mask, image_embeds)

    special_mask = (input_ids == modelf.model.config.image_token_id) & (labels == modelf.model.config.image_token_id)
    for i in range(bs):
        mask_indices = special_mask[i].nonzero(as_tuple=True)[0]
        inputs_embeds[i, mask_indices] = mask_query
        
    if attention_mask is not None:
        attention_mask = attention_mask.to(inputs_embeds.device)

# if we get 4D attention mask we cannot calculate rope deltas anymore.
if position_ids is None and (attention_mask is None or attention_mask.ndim == 2):
    # calculate RoPE index once per generation in the pre-fill stage only
    if (cache_position is not None and cache_position[0] == 0) or modelf.model.rope_deltas is None:
        position_ids, rope_deltas = modelf.model.get_rope_index(
            input_ids, image_grid_thw, video_grid_thw, attention_mask
        )
        modelf.model.rope_deltas = rope_deltas
    # then use the prev pre-calculated rope-deltas to get the correct position ids
    else:
        batch_size, seq_length, _ = inputs_embeds.shape
        delta = cache_position[0] + modelf.model.rope_deltas if cache_position is not None else 0
        position_ids = torch.arange(seq_length, device=inputs_embeds.device)
        position_ids = position_ids.view(1, -1).expand(batch_size, -1)
        if cache_position is not None:  # otherwise `deltas` is an int `0`
            delta = delta.repeat_interleave(batch_size // delta.shape[0], dim=0)
        position_ids = position_ids.add(delta)
        position_ids = position_ids.unsqueeze(0).expand(3, -1, -1)

In [23]:
with torch.no_grad():
    outputs = modelf.model.model(
        input_ids=None,
        position_ids=position_ids,
        attention_mask=attention_mask,
        past_key_values=past_key_values,
        inputs_embeds=inputs_embeds,
        use_cache=use_cache,
        output_attentions=output_attentions,
        output_hidden_states=output_hidden_states,
        return_dict=return_dict,
        cache_position=cache_position,
    )

In [24]:
hidden_states = outputs.last_hidden_state
mask_query = hidden_states[special_mask].view(bs, num_image_token, modelf.model.config.hidden_size)

In [25]:
modelf.model.device

device(type='cuda', index=0)

In [26]:
mask_head = MaskHead(modelf.model.config.hidden_size).to(modelf.model.device)
preds = mask_head(mask_query, image_grid_thw[-1])

In [31]:
preds.shape

torch.Size([2, 1, 784, 784])

In [29]:
hidden_size = modelf.model.config.hidden_size
mlp = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1),
        ).to(modelf.model.device)
t, patch_h, patch_w = image_grid_thw[-1]
mask_hidden_states = mlp(mask_query)
mhs = mask_hidden_states.view(bs, 1, patch_h//2, patch_w//2)

mask_hidden_states.shape

torch.Size([2, 784, 1])

In [30]:
mhs.shape

torch.Size([2, 1, 28, 28])

In [39]:
patch_h

tensor(56, device='cuda:0')

In [30]:
((input_ids == modelf.model.config.image_token_id) & (labels == modelf.model.config.image_token_id)
).shape

torch.Size([2, 1788])

In [ ]:
input_ids.shape

In [ ]:
processorf

In [ ]:
inputs_embeds[0, mask_indices] = mask_query

In [ ]:
inputs_embeds[0, ]

In [ ]:
# This will give you the actual number of True elements
num_true_elements = image_mask.sum().item()
print(f"Number of True elements: {num_true_elements}")
# This should perfectly match the total number of elements in image_embeds
num_embed_elements = image_embeds.numel() # or len(image_embeds.flatten())
print(f"Number of image elements: {num_embed_elements}")

In [ ]:
inputs_embeds.shape

In [ ]:
image_embeds.shape

In [ ]:
pixel_values.shape

In [ ]:
image_grid_thw

In [ ]:
input_ids.shape

In [ ]:
inputs_embeds.shape

In [ ]:
(input_ids == modelf.model.config.image_token_id).shape

In [ ]:
attention_mask.ndim

In [ ]:
mask_indices.shape

In [ ]:
pixel_values.shape

In [ ]:
input_ids.shape

In [ ]:
list(batchf[0]['input_ids'].shape)

In [ ]:
modelf.config

In [ ]:
processorf

In [ ]:
from models.processing_qwen2_vl import Qwen2VLProcessor
from models.image_processing_qwen2_vl import Qwen2VLImageProcessor
from models.processing_qwen2_vl import Qwen2VLProcessor
from models.image_processing_qwen2_vl import Qwen2VLImageProcessor
from models.pilot_llm import PilotLLM

from PIL import Image
import requests
import torch
from torchvision import io
from typing import Dict
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor

from peft import get_peft_model, LoraConfig, PeftModel


processor = Qwen2VLProcessor.from_pretrained("/storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/pilot_llm/weights/Qwen2-VL-2B-Instruct", 
                                                # image_processor=image_processor, 
                                                padding_side="right")
model = PilotLLM.from_pretrained("/storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/pilot_llm/weights/Qwen2-VL-2B-Instruct")

In [ ]:
import os

model = PeftModel.from_pretrained(model, "/storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/pilot_llm/weights/pretrained_Qwen2-VL-2B-Instruct")
weights = torch.load(os.path.join("/storage/project/r-cj124-0/sibidapo3/8750/aeroduo_ws/aeroduo/pilot_llm/weights/pretrained_Qwen2-VL-2B-Instruct", "pytorch_model/mp_rank_00_model_states.pt"))
model.load_state_dict(weights['module'], strict=False)

# If loading a trained main model, do not rewrite a set of lora
model = model.merge_and_unload()

lora_config = LoraConfig(
    r=8,
    target_modules=['down_proj','o_proj','k_proj','q_proj','gate_proj','up_proj','v_proj'],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

for n, p in model.named_parameters():
    if any(
        [
            x in n
            for x in ["pooling_head", "depth_head", 
                        "seg_head", "seg_query", 
                        "mask_head", "coord_query"]
        ]
    ):
        print("n: ", n, "p.shape: ", p.shape)

In [ ]:
model.num_image_token

In [ ]:
model.config.hidden_size

In [ ]:
list(batchq[0].keys())

In [ ]:
model.depth_head

In [ ]:
model.named_parameters

In [ ]:
model.print_trainable_parameters()

In [ ]:
processorq

In [ ]:
batchq

In [ ]:
import os
import json
from PIL import Image
from scipy.ndimage import gaussian_filter

import cv2
import numpy as np
import torch

from vision_process import process_vision_info
from orthography import Orthophoto
from dataset.vis_data import visualize_waypoints

video_frame_num = 5
target_interval = 30
data_list_json_paths = ["../data/train_data_sample.json"]
visualize = False
sigma = 20
visual_prompt = True

data_list = []
for file in data_list_json_paths:
    with open(file, "r") as f:
        datum = json.load(f)
    data_list += datum

data = data_list

ortho_processor = Orthophoto(granularity=0.3)

idx = 5


def preprocess(image, pad_color=(0, 0, 0)):
    img_size = 784
    h, w = image.shape[:2]
    scale = img_size * 1.0 / max(h, w)
    new_h, new_w = h * scale, w * scale
    new_w = int(new_w + 0.5)
    new_h = int(new_h + 0.5)
    resized_image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    resized_hw = (new_h, new_w)

    pad_h = img_size - new_h
    pad_w = img_size - new_w
    padded_image = cv2.copyMakeBorder(resized_image, 0, pad_h, 0, pad_w,
                                    cv2.BORDER_CONSTANT, value=pad_color)
    return padded_image, resized_hw

def generate_prob_message_v2(pil_image, description):
    visual_prompt = True
    if isinstance(pil_image, np.ndarray):
        pil_image = Image.fromarray(pil_image)
    if isinstance(description, list):
        description = description[0]
    text_parts = description.split("The description of the target and its surrounding is shown below.")
    direction = text_parts[0].strip().split("Compass north corresponds to the top of the bird's-eye-view image.")[-1]
    direction = direction.strip()
    object_description = text_parts[-1].strip()

    prob_message = [{
        'role':'user',
        'content':[
            {'type':'image', 'image':pil_image},
            {'type':'text', 'text': "Task: Predict the probability distribution of the drone's future flight locations to search for the target."
            "Input Image: The image is an orthophoto map generated from the drone's past flight trajectory."},
            {'type':'text', 'text': "The green dots indicate past drone positions." if visual_prompt else " "},
            {'type':'text', 'text': "The top of the image corresponds to the north in the world coordinate system.\n" 
            f"Target Information: {direction}"
            f"The description of the target and its surrounding is shown below: {object_description}\n"
            "Objective: "
            "Analyze the provided orthophoto map and target information."
            "Predict the next flight location for the drone that maximize the probability of finding the target."
            "Output a probability map, indicating the likelihood of different regions in the orthophoto map being the optimal next flight destinations."
            }
        ]
    },
    {
        'role':'assistant',
        'content':[
            {'type':'image', 'image':pil_image}
        ]
    }
    ]
    return prob_message
    
def gaussian(target, img_size):
    sigma = 20
    h, w = img_size
    prob_map = np.zeros((h, w), dtype=np.float32)
    i, j = target
    i, j = min(round(i),h-1), min(round(j),w-1)
    prob_map[i, j] = 1
    i_1 = min(h-1, i+1)
    j_1 = min(w-1, j+1)
    i_2 = max(0, i-1)
    j_2 = max(0, j-1)
    prob_map[i, j_1] = 1
    prob_map[i, j_2] = 1
    prob_map[i_1, j] = 1
    prob_map[i_2, j] = 1

    sigma = max(h,w) // 25 if sigma is None else sigma
    prob_map = gaussian_filter(prob_map, sigma=sigma)
    return prob_map

def get_prob_map(ortho, coor_map, end, ortho_depth=None, delta_height=None):
    h, w = ortho.shape[:2]
    if ortho_depth is not None:
        depth_mask = (ortho_depth > delta_height).reshape(h,w)
    i, j = ortho_processor.world_to_pixel(end, coor_map=coor_map)
    i, j = min(max(round(i), 0), h-1), min(max(round(j), 0), w-1)
    prob_map_0 = np.zeros((h, w))
    prob_map_0[i, j] = 1

    yy, xx = np.ogrid[:h, :w]
    distances = np.sqrt((yy - i)**2 + (xx - j)**2)
    max_dist = np.sqrt(2*(h-1)**2)
    prob_map_0 = 1 - distances / max_dist
    prob_map_0[~depth_mask] = 0

    new_i, new_j = np.unravel_index(np.argmax(prob_map_0), (h, w))
    prob_map = gaussian((new_i, new_j), (h, w))
    # prob_map[~depth_mask] = 0
    prob_map = prob_map / (np.max(prob_map) + 1e-6)

    return prob_map, depth_mask

data_info = data[idx]
traj_dir = data_info["traj_folder_path"]
depth_dir = os.path.join(traj_dir, "bevcamera_depth")
image_dir = os.path.join(traj_dir, "bevcamera")
log_dir = os.path.join(traj_dir, "log")
image_path = data_info["image_path"]

high_uav_pos_now = data_info["high_uav_pos_now"]
end_pos = data_info["end_pos"]
int_time = data_info["int_time"]
target_time = int_time+target_interval

description_path = os.path.join(traj_dir, "object_description_with_help.json")
with open(description_path, 'r') as f:
    description = json.load(f)
description = description[0]

image_files = sorted([f for f in os.listdir(image_dir)])
image_numbers = sorted([int(f.split('.')[0]) for f in image_files])
available_images = [t for t in image_numbers if t <= int_time]
available_num = len(available_images)
if available_num > video_frame_num:
    indices = [round(i * (available_num - 1) / (video_frame_num - 1)) for i in range(video_frame_num)]
    available_images = [available_images[i] for i in indices]
names = [f"{t:06d}" for t in available_images]

# historial orthography
frame_paths = [os.path.join(image_dir, f"{idx}.png") for idx in names]
log_paths = [os.path.join(log_dir, f"{idx}.json") for idx in names]
depth_paths = [os.path.join(depth_dir, f"{idx}.png") for idx in names]
positions = np.array([
            json.load(open(log_path, "r"))["sensors"]["state"]["position"] for log_path in log_paths
            ])
frames = np.array([cv2.imread(frame_path) for frame_path in frame_paths])
depths = np.array([cv2.imread(depth_path, cv2.IMREAD_UNCHANGED) for depth_path in depth_paths])

coord_3d_clouds = ortho_processor.project_images_to_3d(depths, positions)
merged_ortho, coor_map, merged_depth = ortho_processor.orthorectify(frames, coord_3d_clouds, depths)
merged_ortho = cv2.cvtColor(merged_ortho, cv2.COLOR_BGR2RGB)


with open(os.path.join(traj_dir, "gt_waypoints.json"), "r") as f:
    gt_waypoints = json.load(f)
if len(gt_waypoints) > len(image_numbers):
    indices = [round(i * (len(gt_waypoints) - 1) / (len(image_numbers) - 1)) for i in range(len(image_numbers))]
else:
    indices = [i for i in range(len(gt_waypoints))]
index = image_numbers.index(int_time)
time_now = indices[index]
time_target = time_now + target_interval
waypoint_now = gt_waypoints[time_now]
if time_target < len(gt_waypoints):
    waypoint_target = gt_waypoints[time_target]
else:
    waypoint_target = end_pos
if visual_prompt:
    time_indexs = indices[:index]
    vis_waypoints = [gt_waypoints[n] for n in time_indexs]
    merged_ortho_prob = visualize_waypoints(vis_waypoints, coor_map, merged_ortho)
else:
    merged_ortho_prob = merged_ortho
prob_map, depth_mask = get_prob_map(merged_ortho, coor_map, waypoint_target, 
                                    ortho_depth=merged_depth, delta_height=waypoint_now[2]-high_uav_pos_now[2])
ortho_resize_pad, resized_hw = preprocess(merged_ortho_prob)
prob_map, _ = preprocess(prob_map)
depth_mask = depth_mask.astype(np.uint8)
depth_mask, _ = preprocess(depth_mask)
prob_message = generate_prob_message_v2(ortho_resize_pad, description)

prob_map = torch.from_numpy(prob_map)
prob_map = prob_map.unsqueeze(0)


batch = {
            "prob_message": prob_message,
            "prob_map": prob_map.float(),
            "target_time": target_time,
            "traj_dir": traj_dir,
        }

batch['traj_dir']

In [ ]:
masks_list = []
messages_list = []
traj_folders = []
target_times = []

for data in [batch]:
    masks_list.append(data["prob_map"])
    message = data["prob_message"]
    messages_list.append(message)
    traj_folders.append(data["traj_dir"])
    target_times.append(data["target_time"])

texts = [processorq.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in messages_list]
image_inputs, video_inputs = process_vision_info(messages_list)

inputs = processorq(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

input_ids_lists = inputs['input_ids'].tolist()
assert len(messages_list) == len(input_ids_lists)

labels = []
for ids_list in input_ids_lists:
    label_ids = [-100] * len(ids_list) # -100 is the ignore index in loss function
    for begin_end_indexs in find_assistant_content_sublist_indexes(ids_list):
        label_ids[begin_end_indexs[0]:begin_end_indexs[1]] = ids_list[begin_end_indexs[0]:begin_end_indexs[1]]
    labels.append(label_ids)
labels = torch.tensor(labels, dtype=torch.int64)

In [ ]:
labels

In [ ]:
texts

In [ ]:
processorq.tokenizer.encode("<|im_start|>assistant\n")

In [ ]:
processorq.tokenizer.encode("<|im_end|>\n")

In [ ]:
batchq[1]['labels'][0][-1000:]

In [ ]:
processorq